# Run_Eval -- grade the conversations, write the score lake

**This is the one place scores are produced.** Every other module in `eda_analysis` reads finished
artifacts; this notebook makes them. It walks each arm's conversations on disk, asks a JUDGE to
fill every rubric, and writes one parquet per `(judge, rep, metric, arm, model state)`:

```
data/eval_scores/judge=<tag>/rep=<r>/metric=<M>/<EXPERIMENT_NAME>/model_iter_<N>.parquet
```

96 rows each -- one per persona -- with `persona_id` in every row. That column is the only valid
pairing key downstream; never pair on file order or row order.

### Re-running costs nothing

A partition is **done** when its parquet exists and holds a row for every conversation currently on
disk for that model state, and `discover_scorable` skips exactly those. So a second pass over a
finished lake issues **zero** grader calls, an interrupted pass resumes at the next missing file,
and a partition that errored simply stays in the plan until it succeeds. Resume granularity is one
whole parquet: a partially-scored state is re-scored in full, which is the price of never having to
reconcile a half-written file. (Writes go through a temp file plus `os.replace`, so a killed process
leaves either the previous complete file or no file.)

### With the default LOCAL judge, a full re-score is FREE

The default judge is the same `google/gemma-4-E2B-it` behind vLLM that plays the oracle and the
patient, so the entire grid costs GPU-hours and **$0 in API**. That is the point of Exp4, and it is
the thing that was impossible in Exp3: there the lake was several hundred dollars of irreplaceable
vendor calls, and "delete the partition and re-run it" was not a move anyone could make. Here it
is -- drop `judge=<tag>/rep=<r>/metric=<M>/` and re-run this notebook.

The freedom ends the moment `JUDGE_PROVIDER` names a vendor API. Then the lake is expensive again
and Exp3's rules apply.

### This notebook is NOT a family and writes nothing under `results/`

There is no `EdaConfig` and no `notebook_setup` here, and `tools/render_results.py` never executes
it (`notebooks/scoring/` is exempt from the `FAMILIES` map on purpose). Its only output is the score
lake under `data/`. **Do not look for its figures or tables -- it has none.**

Cell order below is deliberate: **setup -> configure -> serve -> sanity gate -> plan -> run ->
verify**. The gate sits before the plan because a degenerate grader is invisible after the fact.

## 1. Setup

`from eda_analysis import scoring` is written out explicitly, and that is the whole design:
`scoring` is deliberately absent from the package's lazy attribute map, so no analysis import can
reach the one module that talks to a grader by accident. **This import is the moment someone chooses
to spend** (GPU-hours on the open stack, real money on a vendor judge).

Importing `eda_analysis` also prepends `Exp4_OpenStack/code` to `sys.path`, which is why `roles`,
`tools.vllm_serve` and `tools.oracle_sanity` resolve below to the single canonical copies the
trainers themselves run.

In [ ]:
import os
import sys

# Find eda/ -- the directory holding eda_analysis/ -- from wherever this notebook was opened.
_eda_dir = os.path.abspath(os.getcwd())
while _eda_dir != os.path.dirname(_eda_dir) and not os.path.isdir(
        os.path.join(_eda_dir, "eda_analysis")):
    _eda_dir = os.path.dirname(_eda_dir)
if _eda_dir not in sys.path:
    sys.path.insert(0, _eda_dir)

import pandas as pd
from IPython.display import display

# EXPLICIT on purpose -- see the note above. `data` is the read side; `scoring` is the write side.
from eda_analysis import data, scoring
from eda_analysis.constants import DATA_DIR, EVAL_SCORES_DIR, N_PERSONAS, available_judge_tags

# Canonical trainer-side modules, reachable because importing eda_analysis put code/ on sys.path.
from roles import DEFAULT_JUDGE_MODEL, make_binding, reset_client_cache
from tools import oracle_sanity
from tools.vllm_serve import ensure_alive, report_weights_gib, serve_roles

print("eda root   :", _eda_dir)
print("data root  :", DATA_DIR)
print("score lake :", EVAL_SCORES_DIR)
print("judges already in the lake:", ", ".join(available_judge_tags()) or "(none yet)")
print("arms with conversations on disk:", len(data.discover_arms()))

## 2. Configuration

Every knob this notebook has. Nothing else configures a scoring pass.

The judge is the one setting that must never be implicit: `JUDGE_MODEL` decides the `judge=<tag>`
directory every score is filed under, and the tag is derived *inside* `score_model_state` from the
binding that actually does the grading -- so the folder cannot disagree with the grader that filled
it. `run_scoring` additionally refuses a plan whose `judge` column does not match the binding it was
handed.

`REP = 0` is the full-grid draw every family reads. Use `rep >= 1` only for repeatability re-draws;
a mismatch would overwrite the draw everything else reports.

`METRICS = None` means all eight stored instruments (`Q1`, `Q2`, `WAI_SR`, `CSQ8`, `MI_SAT`, `MITI`,
`PCT`, `MICI`). `Q1Q2` is a **composite** -- the loader averages Q1 and Q2 after reading, exactly as
`core.oracle.score_conversation` does -- so it is never scored and never stored; asking for it
raises.

In [ ]:
# ---------------------------------------------------------------------------
# JUDGE -- who grades. Decides the judge=<tag> partition; never leave it implicit.
# ---------------------------------------------------------------------------
JUDGE_MODEL    = DEFAULT_JUDGE_MODEL   # "google/gemma-4-E2B-it" -- the open stack's grader
JUDGE_PROVIDER = "openai_compat"       # "openai_compat" = a local server | "openai" = BILLS
JUDGE_BASE_URL = ""                    # "" -> filled in by the serve cell below

# ---------------------------------------------------------------------------
# LOCAL SERVER (openai_compat only; ignored for a vendor judge)
# ---------------------------------------------------------------------------
SERVE_LOCAL = True                     # False = a server someone else started; set JUDGE_BASE_URL
SERVE_PORT = 8000
# vLLM PRE-ALLOCATES this fraction of the card and never gives it back. Grading on an otherwise
# idle GPU wants it HIGH; if a trainer is sharing the card, drop it to ~0.25 and start the server
# FIRST. Applies only when the cell below actually LAUNCHES a server -- an adopted one keeps its own.
SERVE_GPU_MEMORY_UTILIZATION = 0.85
# Do NOT lower this. Measured on 192 real Exp3 transcripts, full Q2 oracle prompts reach 10,042
# tokens; at 8192 about 2.1% of Q2 and 1.0% of Q1 prompts would not fit -- and those are the
# LONGEST conversations, whose length varies by arm and by K. The dropout would be arm-dependent
# bias on the headline metric, and it would be silent, because an unscoreable conversation is
# simply absent rather than an error. Give memory back via SERVE_GPU_MEMORY_UTILIZATION instead.
SERVE_MAX_MODEL_LEN = 16384

# ---------------------------------------------------------------------------
# WHAT TO SCORE
# ---------------------------------------------------------------------------
REP = 0                                # 0 = the full-grid draw every family reads
METRICS = None                         # None = all eight stored instruments; or e.g. ["Q1", "Q2"]

ARM_METHODS = None                     # e.g. ["PTO"]      -- None = no filter
ARM_KS = None                          # e.g. [0]
ARM_MODES = None                       # e.g. ["greedy"]; "" selects GRPO arms
ARM_LABELS = None                      # e.g. ["GRPO_LA5"] -- a DISPLAY key, can match >1 arm

# ---------------------------------------------------------------------------
# HOW HARD TO TRY (per grading call)
# ---------------------------------------------------------------------------
CONCURRENCY       = scoring.DEFAULT_CONCURRENCY      # in-flight calls WITHIN one model state
MAX_TOKENS        = scoring.JUDGE_MAX_TOKENS         # clipped JSON -> retries -> holes; raise first
MAX_RETRIES       = scoring.JUDGE_MAX_RETRIES        # attempts per call, first one included
REQUEST_TIMEOUT   = scoring.JUDGE_REQUEST_TIMEOUT    # PER ATTEMPT, seconds
MIN_SUCCESS_RATIO = scoring.MIN_SUCCESS_RATIO        # below this a partition is NOT written at all

# ---------------------------------------------------------------------------
# THE SANITY GATE (cell 4)
# ---------------------------------------------------------------------------
SANITY_QUICK = False                   # False = all 12 fixture transcripts; True = the 2 extremes
SANITY_QUESTIONNAIRES = (1, 2)         # the fixture carries reference scores for Q1/Q2 only
# Match the scoring pass: a gate run at a gentler concurrency can pass against a server that
# falls over under the real load.
SANITY_CONCURRENCY = CONCURRENCY
SANITY_REPORT_PATH = ""                # "" = do not archive; a dir gets oracle_sanity.json

# ---------------------------------------------------------------------------
# RUN
# ---------------------------------------------------------------------------
DRY_RUN = False                        # True = validate the plan, make no calls

print(f"judge      : {JUDGE_PROVIDER}:{JUDGE_MODEL}")
print(f"rep        : {REP}")
print(f"metrics    : {'all eight stored' if METRICS is None else ', '.join(METRICS)}")
print(f"call policy: concurrency={CONCURRENCY} max_tokens={MAX_TOKENS} "
      f"max_retries={MAX_RETRIES} timeout={REQUEST_TIMEOUT:.0f}s")
if JUDGE_PROVIDER != "openai_compat":
    print("\n*** WARNING: this is a VENDOR judge. Every call below is BILLED, and a re-score is "
          "no longer free. Check the plan cell's cost line before running. ***")

## 3. Serve the judge

`serve_roles` is **idempotent**: a healthy server already listening on the port and serving the
right model is *adopted*, not duplicated, so re-running this cell is safe. A server answering on
that port with a **different** model is a hard error rather than an adoption -- silently grading the
whole lake with the wrong model is the most expensive failure available here.

Nothing is started for a vendor judge, and nothing is started when `SERVE_LOCAL = False` (then
`JUDGE_BASE_URL` must name the endpoint).

> An `openai_compat` binding with **no** `base_url` does not fail -- the OpenAI SDK quietly points
> at `api.openai.com`, billing a run that exists to cost $0 while the lake records the *local*
> model's tag. `scoring.judge_binding_for` warns about it and `score_model_state` refuses outright;
> this cell fills the URL in from the server it just brought up, so the case should never arise.

In [ ]:
SERVER_HANDLES = {}
_base_url = (JUDGE_BASE_URL or "").strip() or None

if JUDGE_PROVIDER != "openai_compat":
    print(f"[Run_Eval] {JUDGE_PROVIDER} judge -- nothing to serve.")
elif not SERVE_LOCAL:
    if not _base_url:
        raise ValueError(
            "SERVE_LOCAL is False, so JUDGE_BASE_URL must name the running server. An "
            "'openai_compat' binding with no base_url sends every call to api.openai.com -- a "
            "vendor bill on a $0 experiment, filed under the local model's judge tag."
        )
    print(f"[Run_Eval] using an externally managed server at {_base_url} (starting nothing)")
else:
    _wired, SERVER_HANDLES = serve_roles(
        {"judge": make_binding("openai_compat", JUDGE_MODEL, base_url=_base_url)},
        base_port=SERVE_PORT,
        gpu_memory_utilization=SERVE_GPU_MEMORY_UTILIZATION,
        max_model_len=SERVE_MAX_MODEL_LEN,
    )
    _base_url = _wired["judge"].base_url
    for _handle in SERVER_HANDLES.values():
        _weights = report_weights_gib(_handle)
        print(f"[Run_Eval] {_handle.model}: weights "
              f"{'unknown (startup log not parsed)' if _weights is None else f'{_weights:.2f} GiB'}"
              f", max_model_len={_handle.spec.max_model_len}")

JUDGE = scoring.judge_binding_for(
    JUDGE_MODEL,
    provider=JUDGE_PROVIDER,
    base_url=_base_url,
    request_timeout=REQUEST_TIMEOUT,
    max_retries=MAX_RETRIES,
)
JUDGE_TAG = scoring.judge_tag(JUDGE)

print(f"[Run_Eval] judge     : {JUDGE.provider}:{JUDGE.model}")
print(f"[Run_Eval] endpoint  : {JUDGE.base_url or '(provider default)'}")
print(f"[Run_Eval] writes to : data/eval_scores/judge={JUDGE_TAG}/rep={REP}/metric=<M>/...")

## 4. SANITY GATE -- stop here if the grader is not a measuring instrument

**Do not skip this cell.** An open-weights grader fails in two ways and only one of them is loud:

1. **LOUD** -- it ignores the schema or returns the wrong number of item scores. `core.oracle`'s
   validation ladder catches it, but "caught" is not "visible": what a run sees is a rising retry
   count and then *biased missingness*, because the conversations the grader found hardest are
   exactly the ones that disappear. Hence the gate demands `schema_valid_rate == 1.000`, not 0.98.
2. **SILENT, AND WORSE** -- it honours the schema perfectly and returns degenerate scores: every
   item a 4, near-zero variance across conversations a validated grader placed four points apart.
   That parses, passes every rung of the ladder, writes valid parquet, and yields a grader that
   cannot tell any two arms apart. Nothing downstream flags it. The contrast tables come back at
   ~0, the plots look flat, and it reads like a finding.

A four-arm grid of ten model states each is `4 x 10 x 8 x 96 = 30,720` graded cells. Producing all
of them with a degenerate grader and discovering it in the contrast tables is the expensive path;
this gate is 12 transcripts x 2 rubrics = 24 calls.

**Hard gates (block the run):** perfect schema compliance on every requested rubric, and
per-conversation SD at or above `MIN_SCORE_SD` on every rubric *and* on the pooled reward.
**Soft (reported, never blocking):** Spearman rank agreement with the frozen `gpt-4o-mini`
reference, and the level offset. Exp3 and Exp4 are **not on the same score axis** -- a different
grader sits systematically higher or lower, so an offset of a point or more is expected and says
nothing about fitness. What survives the axis change is the ORDERING and the SPREAD.

The gate calls `core.oracle.get_evaluation_json` -- the same prompt builder, schema shim, validation
ladder and retry policy this notebook's scoring path uses. A green report against a private scorer
would prove nothing about the run.

Widening `SANITY_QUESTIONNAIRES` beyond `(1, 2)` is legitimate: the hard gates need no reference at
all. The soft columns just come back `n/a` for the six rubrics the fixture has no reference for --
which means "no reference exists", never "the grader failed".

In [ ]:
sanity_report = scoring.run_async(oracle_sanity.run_sanity(
    JUDGE,
    questionnaire_ids=SANITY_QUESTIONNAIRES,
    quick=SANITY_QUICK,
    concurrency=SANITY_CONCURRENCY,
    max_tokens=MAX_TOKENS,          # same budget the scoring pass uses -- a gate at a LARGER
    max_retries=MAX_RETRIES,        # budget tests a configuration this notebook will not run
    request_timeout=REQUEST_TIMEOUT,
    progress=True,
))

print(oracle_sanity.format_report(sanity_report))

if SANITY_REPORT_PATH:
    print("report written to", oracle_sanity.write_report(sanity_report, SANITY_REPORT_PATH))

_passed, _reasons = oracle_sanity.check_gates(sanity_report)
if not _passed:
    raise RuntimeError(
        "ORACLE SANITY FAILED -- refusing to score with this grader:\n  "
        + "\n  ".join(_reasons)
        + "\n\nScoring the grid anyway would write tens of thousands of valid-looking cells that "
          "mean nothing, and nothing downstream would flag it. Check, in order: thinking mode is "
          "off for this binding, MAX_TOKENS is not clipping the JSON, and the pinned vLLM accepts "
          "strict json_schema (core.oracle.set_openai_compat_strict(False) if it 400s on the key). "
          "Then re-run this cell."
    )

print(f"\nSANITY GATE PASSED -- {JUDGE.provider}:{JUDGE.model} honours the schema and separates "
      f"the fixture. Proceed.")

## 5. Plan -- what is missing, and what it will cost

Nothing is scored here. `discover_scorable` lists one row per `(arm, model state, metric)` still
missing from the lake, and `estimate_calls` says what running it implies -- `$0 (local)` for the
open stack, a labelled projection for a vendor judge, and `unknown` (never `$0`) for a vendor judge
with no pricing row.

An **empty plan is the normal, correct result of a second run.** It is not a failure.

Arms are discovered from `data/conversations/` -- a run becomes scoreable the moment its
conversations land, and there is no registry to edit. One call per `(conversation, rubric)`: each
parquet is one rubric and each rubric is one schema-constrained request.

> A state whose conversation count later GROWS -- a repair pass that regenerates two failed personas
> -- re-enters the plan and is re-scored in full, all 96 calls, not two. Free on the local stack;
> check the cost line first on a vendor judge.

In [ ]:
ARMS = data.filter_arms(
    data.discover_arms(),
    methods=ARM_METHODS, ks=ARM_KS, modes=ARM_MODES, arm_labels=ARM_LABELS,
)
METRIC_KEYS = list(scoring.STORED_METRICS) if METRICS is None else list(METRICS)

plan = scoring.discover_scorable(ARMS, judge=JUDGE_TAG, rep=REP, metrics=METRIC_KEYS)
estimate = scoring.estimate_calls(plan, binding=JUDGE, max_retries=MAX_RETRIES)

print(f"arms            : {len(ARMS)}"
      f"  [{', '.join(a.label for a in ARMS) if ARMS else 'none on disk'}]")
print(f"model states    : {sum(len(a.iters) for a in ARMS)}")
print(f"metrics         : {', '.join(METRIC_KEYS)}")
print(f"partitions to do: {estimate['n_partitions']}")
print(f"grading calls   : {estimate['n_calls']:,}"
      f"   (worst case with retries: {estimate['n_calls_worst_case']:,})")
print(f"cost            : {estimate['cost']}")
for _note in estimate["notes"]:
    print(f"  note: {_note}")

if plan.empty:
    print("\nNothing to score: every requested partition already exists for "
          f"judge={JUDGE_TAG}, rep={REP}. Skip to the verify cell.")
else:
    _by_arm = (plan.groupby(["arm_label", "experiment_name"], as_index=False)
                   .agg(states=("state_index", "nunique"),
                        partitions=("metric", "size"),
                        calls=("n_conversations", "sum")))
    print(f"\nMissing partitions by arm (judge={JUDGE_TAG}, rep={REP}):")
    display(_by_arm)

    _by_metric = (pd.Series(estimate["by_metric"], name="calls")
                    .rename_axis("metric").reset_index()
                    .sort_values("metric", kind="mergesort"))
    print("Grading calls by metric:")
    display(_by_metric)

## 6. Run

Model states run **sequentially**; each already fans out to `CONCURRENCY` in-flight calls, so
running two at once would double the load on the server without touching the bound that was
actually configured. Sequential also makes the resume granularity exactly one file.

A state that fails is recorded with `status="error"` and **the run continues** -- one state that
could not be graded should not cost the twenty that could, and the failure comes back in the frame
rather than vanishing. Those partitions stay in the plan, so re-running the previous two cells
retries exactly them.

A partition where fewer than `MIN_SUCCESS_RATIO` of conversations graded successfully is **not
written at all**. That is deliberate: a written file counts as done forever, so a mostly-empty
partition would freeze a broken grader's output into the lake and every downstream table would
silently be computed over the conversations that happened to be easy to score.

> Interrupting this cell is safe. Completed partitions are on disk; re-run the plan cell (the `plan`
> frame is now stale) and then this one.

In [ ]:
for _name, _handle in SERVER_HANDLES.items():
    _restarts_before = _handle.restarts
    ensure_alive(_handle)
    if _handle.restarts != _restarts_before:
        # The base_url is unchanged, so the client cache would hand back a pool of connections to
        # a process that no longer exists -- a burst of connection errors naming nothing.
        print(f"[Run_Eval] {_name} was restarted -- dropped {reset_client_cache()} cached client(s)")

results = scoring.run_async(scoring.run_scoring(
    plan,
    binding=JUDGE,
    rep=REP,
    concurrency=CONCURRENCY,
    dry_run=DRY_RUN,
    progress=True,
    max_tokens=MAX_TOKENS,
    max_retries=MAX_RETRIES,
    request_timeout=REQUEST_TIMEOUT,
    min_success_ratio=MIN_SUCCESS_RATIO,
))

if results.empty:
    print("nothing was scored -- the plan was empty (a re-run costs zero calls by design).")
else:
    print("\n" + results["status"].value_counts().to_string())

    _written = results[results["status"] == "written"]
    if not _written.empty:
        print(f"\nwrote {len(_written)} partition(s), {int(_written['n_rows'].sum()):,} rows, "
              f"in {_written['elapsed_s'].sum() / 60.0:.1f} min")

    _errors = results[results["status"] == "error"]
    if not _errors.empty:
        print(f"\n{len(_errors)} partition(s) FAILED. They stay in the plan; re-run the plan cell "
              f"and this one to retry only these:")
        display(_errors[["arm_label", "model_state", "metric", "error"]])

## 7. Verify

Re-discover the work list: after a successful run it must be **empty**, which is the same test that
makes a second `Run_Eval` cost zero calls. Then read the lake back through the normal analysis
loader and check the shape every downstream family assumes.

Two things worth looking at in the coverage table:

- **personas per scored state.** 96 is a complete state. Fewer means fewer conversations were on
  disk when it was scored -- the partition is complete *for what was there*, and the state re-enters
  the plan (and is fully re-scored) if the missing conversations appear later. On the Google Drive
  symlink, "the directory reads as empty" is **not** proof the conversations are missing: the mount
  can wedge on a single folder and report zero entries while every file is present in Drive. Check
  the cloud before regenerating anything.
- **ungraded rows.** A conversation the judge could not grade is written as a NaN row with
  `oracle_success=False`, on purpose: a visible hole beats an absent one, which would make the
  partition permanently incomplete and hide the fact that it could not be graded at all.

In [ ]:
remaining = scoring.discover_scorable(ARMS, judge=JUDGE_TAG, rep=REP, metrics=METRIC_KEYS)

if DRY_RUN:
    print("DRY_RUN was on -- no calls were made, so the plan is unchanged.")
elif not remaining.empty:
    display(remaining[["arm_label", "model_state", "metric", "n_conversations", "n_rows_existing"]])
    raise AssertionError(
        f"{len(remaining)} partition(s) are still missing after the run. Re-running the plan cell "
        f"and the run cell picks up exactly these -- a completed partition is skipped, so the retry "
        f"costs only what failed. Read the 'error' column in the run cell first."
    )
else:
    print(f"PLAN IS EMPTY: every requested partition exists for judge={JUDGE_TAG}, rep={REP}. "
          f"A re-run of this notebook would now make zero grading calls.")

scores = data.load_scores_long(
    ARMS, METRIC_KEYS, judge=JUDGE_TAG, rep=REP, attach_persona=False, cache=False,
)

if scores.empty:
    print("\nno scores on disk for this judge/rep -- nothing to summarise.")
else:
    _frame = scores.copy()
    _frame["graded"] = (_frame["oracle_success"].fillna(False).astype(bool)
                        if "oracle_success" in _frame.columns else _frame["score"].notna())

    coverage = (_frame.groupby(["arm_label", "model_state", "metric"], as_index=False)
                      .agg(rows=("persona_id", "size"),
                           personas=("persona_id", "nunique"),
                           graded=("graded", "sum"),
                           mean_score=("score", "mean")))
    coverage["ungraded"] = coverage["rows"] - coverage["graded"]

    per_arm = (coverage.groupby("arm_label", as_index=False)
                       .agg(states=("model_state", "nunique"),
                            partitions=("metric", "size"),
                            min_personas=("personas", "min"),
                            max_personas=("personas", "max"),
                            ungraded_rows=("ungraded", "sum")))
    print(f"\nPer-arm coverage (judge={JUDGE_TAG}, rep={REP}; a complete state is "
          f"{len(METRIC_KEYS)} metrics x {N_PERSONAS} personas):")
    display(per_arm)

    _short = coverage[coverage["personas"] != N_PERSONAS]
    if _short.empty:
        print(f"OK: every scored state carries {N_PERSONAS} personas on every metric.")
    else:
        print(f"NOTE: {len(_short)} partition(s) hold fewer than {N_PERSONAS} personas -- that is "
              f"how many conversations were on disk, not a scoring failure. If conversations are "
              f"missing, check the Drive mount before regenerating them.")
        display(_short)

    _ungraded = int(coverage["ungraded"].sum())
    if _ungraded:
        print(f"NOTE: {_ungraded} row(s) across the lake are NaN (oracle_success=False) -- visible "
              f"holes, not absences. A rising count is the early warning for a grader drifting "
              f"off-schema; re-run the sanity gate if it grows.")

## What next

The lake is written. From here:

```powershell
cd Exp4_OpenStack\eda
..\..\.venv\Scripts\python.exe tools\render_results.py       # every family
```

or open a family notebook directly -- `notebooks/arms/outcomes.ipynb`,
`notebooks/lookahead/reward.ipynb`, `notebooks/method/contrast.ipynb`,
`notebooks/compute/cost.ipynb`. They auto-discover arms and read this lake; none of them scores
anything.

**Again: this notebook wrote nothing under `results/`.** Its output is entirely
`data/eval_scores/judge=<tag>/rep=<r>/metric=<M>/<EXPERIMENT_NAME>/model_iter_<N>.parquet`.

To force a clean re-score with the local judge, delete the partition directory and re-run this
notebook -- it costs GPU time and no money. To add a second grader, change `JUDGE_MODEL` and run
again: the scores land in that grader's own `judge=<tag>/` partition and every family can then put
the two side by side.

> **Never average raw scores across judges.** One grader shares a model with the TRAINING oracle --
> the thing the policy was optimized against -- and any other is held out. That is train-vs-test,
> not two raters of one construct, and they do not share a scale. Combine only contrasts (a
> difference between two model states under *one* judge) or standardized quantities.